In [1]:
import pandas as pd
import numpy as np
import re
import math
import os
import glob

In [2]:
area = {'hazard_identification_explanation_area': {'center': (263.5,865.8), 'width': 415.5, 'height': 219.7},
        'mcq_explanation_area': {'center': (263.5, 862.2), 'width': 411.9, 'height': 223.3}, 
        'snl_explanation_area': {'center': (485.9, 864.9), 'width': 849.4, 'height': 217.8},
        'mitigation_explanation_area': {'center': (248.8, 873.2), 'width': 408.2, 'height': 234.3},
       'planning_area': {'center': (1357.2, 423.8), 'width': 347.8, 'height': 184.9},
       'note_area': {'center': (1712.3, 448.5), 'width': 329.5, 'height': 216},
       'cheatsheet_area': {'center': (1536.6, 829.2), 'width': 677.3, 'height': 399.1},
       'timer1': {'center': (1534.8, 205.9), 'width': 84.2, 'height': 31.1},
       'timer2': {'center': (1876.2, 1055.2), 'width': 67.7, 'height': 42.1},
       'timer3': {'center': (413.6,395.4), 'width': 122.6, 'height': 54.9},
       'progress_review_area': {'center': (649.7,396.3), 'width': 298.4, 'height': 60.4}
       }
for a in area.keys():
    left = area[a]['center'][0] - 0.5 * area[a]['width']
    right = area[a]['center'][0] + 0.5 * area[a]['width']
    bottom = area[a]['center'][1] - 0.5 * area[a]['height']
    top = area[a]['center'][1] + 0.5 * area[a]['height']
    area[a]['range'] = [left, right, bottom, top]   

In [3]:
def merge_tables(imotion_log, game_action_log, note_action_log=None):
    df_eye = pd.read_csv(imotion_log, skiprows=5)
    df_eye = df_eye[['Timestamp', 'Datetime', 'Gaze X', 'Gaze Y']]
    df_eye = df_eye.dropna(how='any')
    df_eye['Timestamp'] = df_eye['Timestamp']/1000
    df_eye['Datetime'] = pd.to_datetime(df_eye['Datetime'], errors='coerce')
    df_eye['Datetime'] = df_eye['Datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
    gaze_area = []
    for index, row in df_eye.iterrows():
        x = row['Gaze X']
        y = row['Gaze Y']
        area_name = 'gap'
        area_list = []
        for a in area.keys():
            area_range = area[a]['range']
            if area_range[0] < x < area_range[1] and area_range[2] < y < area_range[3]:
                area_name = a
                area_list.append(area_name)
        if area_name == 'gap':
            area_list.append(area_name)
        gaze_area.append(area_list)
    df_eye['area'] = gaze_area
    date = ''
    df = {'time': [], 'area': []}
    last_area = ''
    time_area = []
    for index, row in df_eye.iterrows():
        if not date:
            date = row['Datetime']
            last_area = row['area']
            time_area += row['area']
        elif date != row['Datetime']:
            df['time'].append(date)
            df['area'].append(time_area)
            time_area = row['area']
            last_area = row['area']
            date = row['Datetime']
        else:
            if last_area != row['area']:
                last_area = row['area']
                time_area += last_area
    df['time'].append(date)
    df['area'].append(time_area)
    df = pd.DataFrame(df)
    df_action = pd.read_excel(game_action_log)
    df_action['answer'] = df_action['answer'].astype(str)
    df_action['time'] = pd.to_datetime(df_action['time'])
    df_action = df_action.sort_values('time')
    df_action = (df_action.groupby('time', as_index=False).agg({'type': ', '.join, 'answer': ', '.join}))
    df['time'] = pd.to_datetime(df['time'])
    df_merged = pd.merge(df_action, df, how='outer', on='time')
    if not note_action_log:
        df_merged['action'] = [""] * len(df_merged['time'])
        df_merged = df_merged.set_index('time')
        df_merged = df_merged.reindex(
            pd.date_range(df_merged.index.min(), df_merged.index.max(), freq='s')
        ).reset_index(names='time')
        return df_merged
    df_note = pd.read_excel(note_action_log)
    df_note = (df_note.groupby('time', as_index=False).agg({'action': ', '.join}))
    df_note['time'] = pd.to_datetime(df_note['time'])
    df_merged = pd.merge(df_merged, df_note, how='outer', on='time')
    df_merged = df_merged.set_index('time')
    df_merged = df_merged.reindex(
        pd.date_range(df_merged.index.min(), df_merged.index.max(), freq='s')
    ).reset_index(names='time')

    return df_merged
       

In [4]:
def label_action(df):
    complete_count = 0
    learning_actions = []
    status = {"game_tutorial": False, "game_site_navigation": False, "game_hint_viewing": False,
              "game_hazard_search": False, "game_hazard_explanation_reading": False , "game_hazard_quiz_answer": False, 
              "game_quiz_feedback_review": False, "game_hazard_identification": False, "game_risk_rating_input": False, 
              "game_risk_rating_review": False, "game_mitigation_option_view": False, "game_mitigation_feedback_review": False, "game_summary_review": False, 
              "game_show_more_reading": False, "game_redo": False, "irrelevant_navigation": False}
    indicators = {
                 "start_guide": [("game_tutorial", True), ("game_site_navigation", True)], 
                 "end_guide": [("game_tutorial", False), ("game_hazard_search", True)], 
                 "press_esc": [("game_site_navigation", True), ("game_hazard_search", False)], 
                 "press_hint": [("game_hint_viewing", True)],
                 "press_checkpoint": [("game_hazard_search", True), ("game_hint_viewing", False), ("game_site_navigation", False)], 
                 "press_hazard": [("game_hazard_search", False)], 
                 "answer_hazard": [("game_hazard_identification", False)],
                 "answer_mcq": [("game_hazard_quiz_answer", False), ("game_quiz_feedback_review", True), ("game_hazard_search", False), ("game_hint_viewing", False)], 
                 "press_mcq_next": [("game_quiz_feedback_review", False), ("game_risk_rating_input", True), ("game_show_more_reading", False)],
                 "press_snl_submit": [("game_risk_rating_input", False), ("game_risk_rating_review", True)],
                 "press_snl_done": [("game_risk_rating_review", False), ("game_show_more_reading", False), ("game_hazard_search", True)],
                 "press_mitigation_done": [("game_mitigation_option_view", True), ("game_mitigation_feedback_review", False), ("game_show_more_reading", False)], 
                 "answer_mitigation_mcq": [("game_mitigation_feedback_review", True), ("game_mitigation_option_view", False), ("game_tutorial", False)],
                 "press_show_more": [("game_show_more_reading", True)],
                 "press_summary_next": [("game_summary_review", False), ("irrelevant_navigation", True)],
                 "press_redo": [("game_redo", True), ("game_summary_review", False)],
                 "press_bonus_qn": [("game_mitigation_option_view", True)],
                }
    order = {"start_guide": 0, "end_guide": 0.1, "start_timer": 0.2, "press_checkpoint": 1, "press_esc": 2, "press_hazard": 3, "answer_hazard": 4, "answer_mcq": 5, "press_mcq_next": 6, 
             "press_snl_submit": 7, "press_snl_done":8, "answer_mitigation_mcq": 9, "press_mitigation_done": 10, "press_complete": 11}
    indicators_ignored = ["start_game"]
    df = df.fillna("")

    def label_game_action(game_action, action, status, complete_count, row):
        if game_action in indicators_ignored:
            for game_status in status:
                if status[game_status] and game_status not in action:
                    action.append(game_status)
            return action, status          
            
        if game_action == "press_complete":
            if complete_count%2 == 1:
                status['game_mitigation_option_view'] = True
                if complete_count == 1:
                    status['game_tutorial'] = True
            else:
                status['game_summary_review'] = True
                status['game_mitigation_option_view'] = False
            status['game_site_navigation'] = False
            for game_status in status:
                if status[game_status] and game_status not in action:
                    action.append(game_status)
            return action, status 

        if game_action == "start_stage":
            if complete_count > 0:
                status['game_redo'] = True     
                status['irrelevant_navigation'] = False
            for game_status in status:
                if status[game_status] and game_status not in action:
                    action.append(game_status)
            return action, status 

        if game_action == "start_timer":
            if complete_count > 0:
                status["game_site_navigation"] = True
            status['game_tutorial'] = False
            for game_status in status:
                if status[game_status] and game_status not in action:
                    action.append(game_status)
            return action, status 
            
        try:
            if game_action:
                for i in indicators[game_action]:
                    status[i[0]] = i[1]
        except:
            print(game_action)
            return None, None
        
        for game_status in status:
            if status[game_status] and game_status not in action:
                action.append(game_status)
        return action, status
    
    for index, row in df.iterrows():
        action = []
        game_action = row['type']
        note_action = row['action']
        area = row['area']

        game_action = game_action.split(", ")
        if len(game_action) == 0:
            for game_status in status:
                if status[game_status]:
                    action.append(game_status)
        elif len(game_action) == 1:
            game_action = game_action[0]
            if game_action == "press_complete":
                complete_count += 1
            action, status = label_game_action(game_action, action, status, complete_count, row)
        else:
            try:
                game_action = sorted(game_action, key=lambda x: order[x])
            except:
                print(game_action)
            for i in game_action:
                if i == "press_complete":
                    complete_count += 1
                action, status = label_game_action(i, action, status, complete_count, row)
      
        if note_action == 'notes':
            action.append('notes_editing')
        if note_action == 'plan':
            action.append('plan_editing')
        
        learning_actions.append(action)
    df['learning_action'] = learning_actions
    return df

In [5]:
def label_extra(df):
    mapping1 = {
                'answer_mcq':'game_hazard_quiz_answer', 
                'press_mcq_next, answer_mcq': 'game_hazard_quiz_answer', 
                'press_esc': 'game_hazard_search', 
                'press_hazard': 'game_hazard_search', 
                'answer_hazard':'hazard_identification_feedback_review', 
                'end_guide, start_timer': 'reading_tutorial',
                'start_timer, end_guide': 'reading_tutorial',
                'end_guide, start_timer': 'reading_tutorial'
               }
    mapping2 = {
                'answer_hazard': 'game_hazard_identification',
                'press_hazard': 'game_hazard_search',
                'press_esc': 'game_hazard_search'
               }
    df = label_with_terminator(df, 'answer_hazard', mapping1)
    df = label_with_terminator(df, 'press_hazard', mapping2)
    return df
    
def label_with_terminator(df, start_signal, mapping):
    df['type_copy'] = df['type'].replace('', np.nan)
    starts = df.index[df['type_copy'] == start_signal]
    group_id = pd.Series([np.nan]*len(df), index=df.index)
    for start in starts:
        end_mask = df.loc[start+1:, 'type_copy'].isin(mapping.keys())
        if end_mask.any():
            end = end_mask.idxmax()
        else:
            continue
        terminator = df.at[end, 'type_copy']
        label = mapping[terminator]
        df.loc[start:end-1, 'learning_action'] = (
            df.loc[start:end, 'learning_action'].apply(lambda lst: lst + [label])
        )
    df = df.drop('type_copy', axis=1)
    return df

In [8]:
def label_eye_tracking(df):
    reading_label = [('mcq_explanation_area', 'mcq_explanation_reading', 'game_quiz_feedback_review'), 
                  ('snl_explanation_area', 'snl_explanation_reading', 'game_risk_rating_review'), 
                  ('mitigation_explanation_area', 'mitigation_explanation_reading', 'game_mitigation_feedback_review'), 
                  ('hazard_identification_explanation_area', 'game_hazard_explanation_reading', 'hazard_identification_feedback_review'), 
                  ('planning_area', 'plan_reading', None), 
                  ('note_area', 'notes_reading', None),
                  ('cheatsheet_area', 'cheatsheet_reading', None)
                 ]
    timers = ['timer1', 'timer2', 'timer3']
    timer_label = 'time_checking'
    progress_review_areas = ['progress_review_area']
    progress_review_label = 'progress_review'
    for label in reading_label:
        df = label_reading(df, label[0], 3, label[1], label[2])
    for timer in timers:
        df['learning_action'] = df.apply(lambda row: row['learning_action'] + [timer_label] if timer in row['area'] else row['learning_action'], axis=1)
    for area in progress_review_areas:
        df['learning_action'] = df.apply(lambda row: row['learning_action'] + [progress_review_label] if area in row['area'] else row['learning_action'], axis=1)
    df['learning_action'] = df.apply(lambda row: [item for item in row['learning_action'] if item != 'game_show_more_reading'] if ('mcq_explanation_reading' not in row['learning_action']) and ("mitigation_explanation_reading" not in row['learning_action']) and ('game_show_more_reading' in row['learning_action']) else row['learning_action'], axis=1)
    return df

def label_reading(df, area, duration, label, condition=None):
    if not condition:
        action = label.split("_")[0] + "_editing"
        df['has_action'] = df['action'].apply(lambda item: item == label.split("_")[0])
        df['action_occured'] = df['has_action'].cummax()
        if area != "cheatsheet_area":
            df['in_area'] = df.apply(lambda row: (row['action_occured']) and (area in row['area']), axis=1)
        else: 
            df['in_area'] = df.apply(lambda row: area in row['area'], axis=1)
        df = df.drop(['has_action'], axis=1)
        df = df.drop(['action_occured'], axis=1)
    else:        
        df['in_area'] = df.apply(lambda row: (area in row['area']) and (condition in row['learning_action']), axis=1)        
    grp = (df['in_area'] != df['in_area'].shift()).cumsum()
    group_size = df.groupby(grp)['in_area'].transform('size')
    mask = df['in_area'] & (group_size >= duration)
    df['learning_action'] = df.apply(lambda row: row['learning_action'] + [label] if mask[row.name] else row['learning_action'], axis=1)
    df = df.drop(['in_area'], axis=1)
        
    return df

In [7]:
def label_tutorial(df):
    df['is_tutorial'] = df.apply(lambda row: 'game_tutorial' in row['learning_action'], axis=1)
    label_list = ['game_site_navigation', 'game_hazard_search', 'game_hazard_identification', 'game_hazard_quiz_answer', 'game_risk_rating_input']
    df['learning_action'] = df.apply(lambda row: [item + '_tutorial' if (item in label_list) and (row['is_tutorial']) else item for item in row['learning_action']], axis=1)
    df = df.drop(['is_tutorial'], axis=1)
    return df

In [10]:
eye_tracking_path = 'D:/research assistant/EEG/imotion data/data_game/eye_tracking/'
game_action_log_path = 'D:/research assistant/EEG/imotion data/data_game/game_action_log/'
note_action_log_path = 'D:/research assistant/EEG/imotion data/data_game/note_action_log/'
count = 0
for filename in os.listdir(eye_tracking_path):
    file_path = os.path.join(eye_tracking_path, filename)
    name = filename.split('_')[0].replace('Subject ', 'p')
    #if name != "p041":
    #    continue
    log_path = game_action_log_path + 'log_*_' + name + '.xlsx'
    try:
        log_file = glob.glob(log_path)[0]
    except:
        print(log_path)
        break
    note_file_path = note_action_log_path + "*_" +name + '.xlsx'
    note_log_file = glob.glob(note_file_path)
    if len(note_log_file) == 0:
        df = merge_tables(file_path, log_file)
    else:
        df = merge_tables(file_path, log_file, note_log_file[0])
    df = label_action(df)
    try:
        df = label_extra(df)
    except:
        print(name)
    df = label_eye_tracking(df)
    df = label_tutorial(df)
    df.to_excel("D:/research assistant/EEG/imotion data/data_game/merged_log/" + name + ".xlsx") 
    #if name == 'p041':
    #    break

C:\Users\jiang_c\AppData\Local\Temp\ipykernel_30292\1457763139.py:2: DtypeWarning: Columns (98,99,104,105) have mixed types. Specify dtype option on import or set low_memory=False.
  df_eye = pd.read_csv(imotion_log, skiprows=5)
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_30292\1457763139.py:2: DtypeWarning: Columns (66,67) have mixed types. Specify dtype option on import or set low_memory=False.
  df_eye = pd.read_csv(imotion_log, skiprows=5)
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_30292\1457763139.py:2: DtypeWarning: Columns (98,99,104,105) have mixed types. Specify dtype option on import or set low_memory=False.
  df_eye = pd.read_csv(imotion_log, skiprows=5)
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_30292\1457763139.py:2: DtypeWarning: Columns (98,99,104,105) have mixed types. Specify dtype option on import or set low_memory=False.
  df_eye = pd.read_csv(imotion_log, skiprows=5)
C:\Users\jiang_c\AppData\Local\Temp\ipykernel_30292\1457763139.py:2: DtypeWarning: Columns (